# ML Assignment 02
### Dataset: House Price Prediction Dataset
### Total Marks: 100

---

## Exam Instructions:
1. প্রথমে নিচের cell এ নিজের **নাম** এবং কোর্সে registration করা **ইমেইল** দিবে
2. Question wise numbering করে Text cell রাখবে এবং এর নিচে Code cell থাকবে, চেষ্টা করবে একটি code cell এ একটি question উত্তর দেওয়ার
3. Google Colab এর মধ্যে কোডগুলো করবে
4. এবং সেই ফাইলটি **'Anyone with the link' & 'View' Access** দিয়ে ফাইলটির Shareable Link টি সাবমিট করবে

---

**Question Dataset Link:** https://www.kaggle.com/datasets/prokshitha/home-value-insights

## Student Information

In [486]:
# Fill in your information
name = "MD. NAFIZ MAHMUD SHAWON"           # Write your full name here
email = "work.nafiz@gmail.com"          # Write your registered email here

print(f"Name  : {name}")
print(f"Email : {email}")

Name  : MD. NAFIZ MAHMUD SHAWON
Email : work.nafiz@gmail.com


In [487]:
# all import

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor, LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error



---
## Question 1 (10 Marks)

Load the House Price dataset and display:
- Dataset shape
- First 10 rows
- 5 random samples

In [488]:
# Question 1
df = pd.read_csv('/content/house_price_regression_dataset.csv')
print(df.shape)
display(df.head(10))
display(df.sample(5))




(1000, 8)


,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Garage_Size,Neighborhood_Quality,House_Price
0,1360,2,1,1981,0.599637,0,5,2.623829e+05
1,4272,3,3,2016,4.753014,1,6,9.852609e+05
2,3592,1,2,2016,3.634823,0,9,7.779774e+05
3,966,1,2,1977,2.730667,1,8,2.296989e+05
4,4926,2,1,1993,4.699073,0,8,1.041741e+06
5,3944,5,3,1990,2.475930,2,8,8.797970e+05
6,3671,1,2,2012,4.911960,0,1,8.144279e+05
7,3419,1,1,1972,2.805281,1,1,7.034131e+05
8,630,3,3,1997,1.014286,1,8,1.738750e+05
9,2185,4,2,1981,3.941604,2,5,5.041765e+05


,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Garage_Size,Neighborhood_Quality,House_Price
632,1399,2,2,2003,1.680543,0,3,310271.263669
652,2253,3,2,2012,3.956577,1,4,551444.631616
457,3794,1,1,1987,2.076860,0,1,783176.636563
164,3431,3,2,1970,4.321306,2,7,762159.391473
784,1372,4,2,2015,2.500377,1,10,377431.064391


---
## Question 2 (10 Marks)

Handle missing values and perform feature engineering:
- Impute missing numerical values using `SimpleImputer` with mean strategy
- Impute missing categorical values using most frequent strategy
- Drop columns with more than 50% missing values
- Perform train-test split with `test_size=0.2` and `random_state=42`

Display the shape of final train and test sets.

In [489]:

df = df.loc[:, df.isnull().mean() <= 0.5]

x = df.drop('House_Price', axis=1)
y = df['House_Price']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

categorical_col = ['Num_Bedrooms', 'Num_Bathrooms', 'Garage_Size', 'Neighborhood_Quality']

categorical_imputer = SimpleImputer(missing_values=np.nan, strategy='most_frequent')
categorical_imputer.fit(x_train[categorical_col])
x_train[categorical_col] = categorical_imputer.fit_transform(x_train[categorical_col])
x_test[categorical_col] = categorical_imputer.transform(x_test[categorical_col])

numerical_col = ['Square_Footage', 'Year_Built', 'Lot_Size']

numerical_imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
numerical_imputer.fit(x_train[numerical_col])
x_train[numerical_col] = numerical_imputer.transform(x_train[numerical_col])
x_test[numerical_col] = numerical_imputer.transform(x_test[numerical_col])

print(x_train.shape)
print(x_test.shape)


(800, 7)
(200, 7)


---
## Question 3 (20 Marks)

Implement **Simple Linear Regression** using **only NumPy** (no Scikit-Learn allowed):
- Compute slope (`m`) and intercept (`c`) using the Batch Gradient Descent
- Predict values for the test set
- Print the learned `m` and `c` values

Use `Square_Footage` as feature (X) and `House_Price` as target (y).

In [490]:
# data scalling first

xtrainmean = x_train['Square_Footage'].mean()
xtrainstd = x_train['Square_Footage'].std()

x_train['scaled_square_footage'] = (x_train['Square_Footage'] - xtrainmean) / xtrainstd
x_test['scaled_square_footage'] = (x_test['Square_Footage'] - xtrainmean) / xtrainstd

x_train = x_train.drop("Square_Footage", axis=1)
x_test = x_test.drop("Square_Footage", axis=1)

ytrainmean = y_train.mean()
ytrainstd = y_train.std()

y_train_scaled = (y_train - ytrainmean) / ytrainstd
y_test_scaled = (y_test - ytrainmean) / ytrainstd

y_train = pd.DataFrame(
    {"House_Price": y_train, "scaled_house_price": y_train_scaled}
)
y_test = pd.DataFrame(
    {"House_Price": y_test, "scaled_house_price": y_test_scaled}
)

y_train = y_train.drop("House_Price", axis=1)
y_test = y_test.drop("House_Price", axis=1)

x_train_arr = np.array(x_train["scaled_square_footage"])
y_train_arr = np.array(y_train["scaled_house_price"]).flatten()

x_test_arr = np.array(x_test["scaled_square_footage"])
y_test_arr = np.array(y_test["scaled_house_price"]).flatten()

# batch gradient descent or gradient descent
def gradient_function(x, y, w, b):
    total_element = x.shape[0]
    prediction = w * x + b
    error = prediction - y
    dj_dw = np.sum(error * x) / total_element
    dj_db = np.sum(error) / total_element
    return dj_dw, dj_db


def gradient_descent(x, y, w_init, b_init, max_iteration, alpha):
    w = w_init
    b = b_init
    for i in range(max_iteration):
        dj_dw, dj_db = gradient_function(x, y, w, b)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
    return w, b


# find m and c
m, c = gradient_descent(x_train_arr, y_train_arr, w_init=0.0, b_init=0.0, max_iteration=1000, alpha=0.01)


# prediction of test data
y_prediction = m * x_test_arr + c
print(y_prediction)
print()
print(f"m is {m:.4f} and c is {c:.4f}\n")



[ 0.94788476 -0.39866383  1.49852978  1.67574887  0.65832144  0.61085204
  1.44314882  1.03886778  0.76512758  1.13064195  0.6298398   0.2010329
 -1.60913348  1.67021077 -0.19691889 -1.52606203 -0.49676725 -0.87731359
  0.40989825 -1.13760413  0.50800168 -0.41132233  0.39723975  1.04677934
 -0.687436   -1.07589391 -0.65658089 -0.71591764 -1.50074502 -0.65895436
 -1.37178649  1.14646508  0.56733842 -1.12336331 -0.54028087  0.62825748
 -0.56243325 -0.18426038 -1.45564909 -1.70328112 -1.17399733 -1.43349671
 -0.47382371 -1.2183021   1.72559174  1.22558074  0.71053777  0.76275411
  1.46925699  0.45657649 -1.42716745  0.43284179  0.6773092   0.32445334
 -0.40182845  0.70262621 -1.72226888 -1.22225789  0.43917105 -1.49916271
  1.42495222 -1.02288641 -1.40422391 -0.5268312  -0.15182296 -0.12334132
 -0.82114147  0.49138739  1.32843111  0.36796695 -0.5030965   0.95579633
 -1.22146673 -0.21036855  0.1116322   0.38932818 -0.44375976 -1.63998859
 -0.11780322  1.12510385 -0.18900732 -1.0268422  -0.

---
## Question 4 (10 Marks)

Build a **ColumnTransformer** that applies:
- `StandardScaler` on numerical columns: `Square_Footage`, `Num_Bedrooms`, `Num_Bathrooms`
- `OneHotEncoder` on categorical column: `Neighborhood_Quality`



In [491]:
df = pd.read_csv('/content/house_price_regression_dataset.csv')


numerical_columns  = ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms']
categorical_columns = ['Neighborhood_Quality']

X = df[["Square_Footage", "Num_Bedrooms", "Num_Bathrooms", "Neighborhood_Quality"]]
y = df["House_Price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

single_transformer = ColumnTransformer (
    transformers = [
        ('categorical_col', OneHotEncoder(sparse_output=False), categorical_columns),
        ('numerical_col', StandardScaler(), numerical_columns)
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)
single_transformer.set_output(transform="pandas")

single_transformer.fit(X_train)
X_train_final = single_transformer.transform(X_train)
X_test_final = single_transformer.transform(X_test)

print(f"Final shape of train data: {X_train_final.shape}")
print(f"Final shape of test data: {X_test_final.shape}")


Final shape of train data: (800, 13)
Final shape of test data: (200, 13)


## Question 5 (20 Marks)

Build a complete **Pipeline** using Scikit-Learn that includes:
- The `ColumnTransformer`
- `SGDRegressor` as the final estimator
- Train the pipeline and evaluate using RMSE and R² score
- Print predicted vs actual values for the first 10 test samples

In [492]:
# Question 5
df = pd.read_csv('/content/house_price_regression_dataset.csv')


numerical_columns  = ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms']
categorical_columns = ['Neighborhood_Quality']

X = df[["Square_Footage", "Num_Bedrooms", "Num_Bathrooms", "Neighborhood_Quality"]]
y = df["House_Price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

single_transformer = ColumnTransformer (
    transformers = [
        ('categorical_col', OneHotEncoder(sparse_output=False), categorical_columns),
        ('numerical_col', StandardScaler(), numerical_columns)
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)
single_transformer.set_output(transform="pandas")

price_pred_pipeline = Pipeline(
    steps=[
        ('preprocessor', single_transformer),
        ('model', SGDRegressor()),
    ]
)

price_pred_pipeline.fit(X_train, y_train)
y_prediction = price_pred_pipeline.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_prediction))
mse = mean_squared_error(y_test, y_prediction)

r2 = r2_score(y_test, y_prediction)

print(f"Root Mean Squared Error (RMSE) : {rmse:.2f}")
print(f"R2 Score                       : {r2:.4f} ({r2*100:.2f}%)")

main_pred_data = pd.DataFrame (
    {
        'main': y_test,
        'pred': y_prediction
    }
)
display(main_pred_data.head(10))

Root Mean Squared Error (RMSE) : 29079.47
R2 Score                       : 0.9869 (98.69%)


,main,pred
521,9.010005e+05,8.514068e+05
737,4.945375e+05,5.077282e+05
740,9.494042e+05,9.857972e+05
660,1.040389e+06,1.022458e+06
411,7.940100e+05,7.598777e+05
678,7.240336e+05,7.658387e+05
626,9.984392e+05,1.003992e+06
513,9.097134e+05,9.034265e+05
859,7.926815e+05,7.971181e+05
136,9.474908e+05,8.889427e+05


---
## Question 6 (20 Marks)

Implement **Multiple Linear Regression** using **Scikit-Learn**:
- The `ColumnTransformer`
- `LinearRegression` as the final estimator
- Train the pipeline and evaluate using RMSE and R² score
- Print predicted vs actual values for the first 10 test samples

In [493]:
# Question 6

lin_reg_pipeline = Pipeline(
    steps=[
        ('preprocessor', single_transformer),
        ('model', LinearRegression()),
    ]
)

lin_reg_pipeline.fit(X_train, y_train)
y_prediction = lin_reg_pipeline.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_prediction))
mse = mean_squared_error(y_test, y_prediction)

r2 = r2_score(y_test, y_prediction)

print(f"Root Mean Squared Error (RMSE) : {rmse:.2f}")
print(f"R2 Score                       : {r2:.4f} ({r2*100:.2f}%)")

main_pred_data = pd.DataFrame (
    {
        'main': y_test,
        'pred': y_prediction
    }
)
display(main_pred_data.head(10))

Root Mean Squared Error (RMSE) : 29093.08
R2 Score                       : 0.9869 (98.69%)


,main,pred
521,9.010005e+05,8.514091e+05
737,4.945375e+05,5.076410e+05
740,9.494042e+05,9.859904e+05
660,1.040389e+06,1.022655e+06
411,7.940100e+05,7.606981e+05
678,7.240336e+05,7.657823e+05
626,9.984392e+05,1.005039e+06
513,9.097134e+05,9.034078e+05
859,7.926815e+05,7.979712e+05
136,9.474908e+05,8.888504e+05


---
## Question 7 (10 Marks) (You have to explore the topic and use the equation via Numpy)
### Dont use LLMs , You can use Documentation

Implement **Multiple Linear Regression** using **only NumPy**:
- Pick random 100 datas from the dataset
- Use the Normal Equation: `θ = (XᵀX)⁻¹ Xᵀy`
- Use `Square_Footage`, `Num_Bedrooms`, and `Num_Bathrooms` as features
- Print the learned coefficients (θ values)

In [494]:
# Question 7
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

ddf = pd.read_csv('/content/house_price_regression_dataset.csv')
ran_df = ddf.sample(100, random_state=42)

x = df[['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms']]
y = df['House_Price']



x_raw = x.values
y_vector = y.values

x_b = np.c_[np.ones((len(x_raw), 1)), x_raw]

theta = np.linalg.inv(x_b.T @ x_b) @ x_b.T @ y_vector

print(f"Intercept (θ₀): {theta[0]:.4f}")
print(f"Square_Footage (θ₁): {theta[1]:.4f}")
print(f"Num_Bedrooms (θ₂): {theta[2]:.4f}")
print(f"Num_Bathrooms (θ₃): {theta[3]:.4f}")


Intercept (θ₀): 5624.6927
Square_Footage (θ₁): 200.8830
Num_Bedrooms (θ₂): 10181.1492
Num_Bathrooms (θ₃): 8730.0002


In [495]:
# sns.boxplot(data=df, x=x["Square_Footage"], y=y)
# plt.show()

In [496]:
# sns.boxplot(data = df, x=x.iloc[:, 1], y=y)
# plt.show()

In [497]:
# sns.boxplot(data = df, x=x.iloc[:, 2], y=y)
# plt.show()